In [2]:
%pwd

'/kaggle/working'

In [3]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

fatal: destination path 'PaddleOCR' already exists and is not an empty directory.


In [4]:
%%writefile /kaggle/working/PaddleOCR/tools/infer/predict_system.py
# Copyright (c) 2020 PaddlePaddle Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import os
import sys

__dir__ = os.path.dirname(os.path.abspath(__file__))
sys.path.append(__dir__)
sys.path.insert(0, os.path.abspath(os.path.join(__dir__, "../..")))

os.environ["FLAGS_allocator_strategy"] = "auto_growth"

import cv2
import copy
import numpy as np
import json
import time
import logging
import tools.infer.utility as utility
import tools.infer.predict_rec as predict_rec
import tools.infer.predict_det as predict_det
import tools.infer.predict_cls as predict_cls
from ppocr.utils.utility import get_image_file_list, check_and_read
from ppocr.utils.logging import get_logger
from tools.infer.utility import (
    get_rotate_crop_image,
    get_minarea_rect_crop,
)

logger = get_logger()


class TextSystem(object):
    def __init__(self, args):
        if not args.show_log:
            logger.setLevel(logging.INFO)

        self.text_detector = predict_det.TextDetector(args)
        self.text_recognizer = predict_rec.TextRecognizer(args)
        self.use_angle_cls = args.use_angle_cls
        self.drop_score = args.drop_score
        if self.use_angle_cls:
            self.text_classifier = predict_cls.TextClassifier(args)
        self.args = args
        
        # Initialize timing statistics
        self.det_time = 0
        self.cls_time = 0
        self.rec_time = 0

    def __call__(self, img, cls=True):
        if img is None:
            logger.debug("no valid image provided")
            return None, None

        ori_im = img.copy()
        
        # Time detection process
        det_start = time.time()
        dt_boxes, _ = self.text_detector(img)
        det_elapsed = time.time() - det_start
        self.det_time += det_elapsed

        if dt_boxes is None:
            return None, None

        img_crop_list = []
        dt_boxes = sorted_boxes(dt_boxes)

        for bno in range(len(dt_boxes)):
            tmp_box = copy.deepcopy(dt_boxes[bno])
            if self.args.det_box_type == "quad":
                img_crop = get_rotate_crop_image(ori_im, tmp_box)
            else:
                img_crop = get_minarea_rect_crop(ori_im, tmp_box)
            img_crop_list.append(img_crop)
        
        # Time classification process (if enabled)
        if self.use_angle_cls and cls:
            cls_start = time.time()
            img_crop_list, _, _ = self.text_classifier(img_crop_list)
            cls_elapsed = time.time() - cls_start
            self.cls_time += cls_elapsed

        # Time recognition process
        rec_start = time.time()
        rec_res, _ = self.text_recognizer(img_crop_list)
        rec_elapsed = time.time() - rec_start
        self.rec_time += rec_elapsed
        
        # Filter by score
        filter_boxes, filter_rec_res = [], []
        for box, rec_result in zip(dt_boxes, rec_res):
            text, score = rec_result[0], rec_result[1]
            if score >= self.drop_score:
                filter_boxes.append(box)
                filter_rec_res.append(rec_result)
                
        return filter_boxes, filter_rec_res
    
    def get_timing_stats(self):
        """Return timing statistics"""
        return {
            'detection_time': self.det_time,
            'classification_time': self.cls_time,
            'recognition_time': self.rec_time,
            'total_time': self.det_time + self.cls_time + self.rec_time
        }


def sorted_boxes(dt_boxes):
    """Sort text boxes in order from top to bottom, left to right"""
    num_boxes = dt_boxes.shape[0]
    sorted_boxes = sorted(dt_boxes, key=lambda x: (x[0][1], x[0][0]))
    _boxes = list(sorted_boxes)

    for i in range(num_boxes - 1):
        for j in range(i, -1, -1):
            if abs(_boxes[j + 1][0][1] - _boxes[j][0][1]) < 10 and (
                _boxes[j + 1][0][0] < _boxes[j][0][0]
            ):
                tmp = _boxes[j]
                _boxes[j] = _boxes[j + 1]
                _boxes[j + 1] = tmp
            else:
                break
    return _boxes


def process_image(text_sys, image_path):
    """Process a single image and return OCR results as JSON"""
    img, flag_gif, flag_pdf = check_and_read(image_path)
    if not flag_gif and not flag_pdf:
        img = cv2.imread(image_path)
    
    if img is None:
        logger.error(f"Failed to load image: {image_path}")
        return {"root": []}
    
    dt_boxes, rec_res = text_sys(img)
    
    if dt_boxes is None or rec_res is None:
        return {"root": []}
    
    # Build result list
    results = []
    for i in range(len(dt_boxes)):
        results.append({
            "transcription": rec_res[i][0],
            "points": np.array(dt_boxes[i]).astype(np.int32).tolist(),
        })
    
    return {"root": results}


def main(args):
    """Main function that returns JSON directly"""
    text_sys = TextSystem(args)
    
    # Warm up if needed
    if args.warmup:
        img = np.random.uniform(0, 255, [640, 640, 3]).astype(np.uint8)
        for i in range(2):  # Reduced from 10 to 2
            _ = text_sys(img)
        # Reset timing after warmup
        text_sys.det_time = 0
        text_sys.cls_time = 0
        text_sys.rec_time = 0
    
    # Process the image
    total_start = time.time()
    result = process_image(text_sys, args.image_dir)
    total_elapsed = time.time() - total_start
    
    # Get timing statistics
    timing_stats = text_sys.get_timing_stats()
    
    # Print timing information
    print("\n" + "="*50)
    print("TIMING STATISTICS")
    print("="*50)
    print(f"Detection time:       {timing_stats['detection_time']:.4f} seconds")
    if text_sys.use_angle_cls:
        print(f"Classification time:  {timing_stats['classification_time']:.4f} seconds")
    print(f"Recognition time:     {timing_stats['recognition_time']:.4f} seconds")
    print(f"Total inference time: {timing_stats['total_time']:.4f} seconds")
    print(f"Overall time:         {total_elapsed:.4f} seconds")
    print("="*50 + "\n")
    
    # Save to JSON file
    output_path = "ocr_results.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    logger.info(f"OCR completed: {len(result['root'])} text regions found")
    logger.info(f"Results saved to {output_path}")
    
    return result


if __name__ == "__main__":
    args = utility.parse_args()
    result = main(args)

Overwriting /kaggle/working/PaddleOCR/tools/infer/predict_system.py


In [5]:
!python -m pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!pip install -q paddleocr --user
!pip install -q imutils --user
!pip install -q pyclipper lmdb rapidfuzz paddleocr opencv-python --user

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu129/
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cuda-nvrtc-cu12/nvidia_cuda_nvrtc_cu12-12.9.41-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl (89.6 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cuda-runtime-cu12/nvidia_cuda_runtime_cu12-12.9.37-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.5 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cuda-cupti-cu12/nvidia_cuda_cupti_cu12-12.9.19-py3-none-manylinux_2_25_x86_64.whl (10.8 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cudnn-cu12/nvidia_cudnn_cu12-9.9.0.52-py3-none-manylinux_2_27_x86_64.whl (781.4 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cublas-cu12/nvidia_cublas_cu12-12.9.0.13-py3-none-manylinux_2_27_x86_64.whl (580.7 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu129/nvidia-cufft-cu12/nvidia_cufft_c

In [6]:
!pwd
import os
import cv2
import json

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from imutils import perspective

# from paddleocr import PaddleOCR

/kaggle/working


In [7]:
import os
import paddle

!pip show paddleocr
print("GPU available:", paddle.device.is_compiled_with_cuda())

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Name: paddleocr
Version: 3.4.1
Summary: Awesome multilingual OCR and document parsing toolkits based on PaddlePaddle
Home-page: https://github.com/PaddlePaddle/PaddleOCR
Author: 
Author-email: PaddlePaddle <paddleocr@baidu.com>
License: Apache License 2.0
Location: /root/.local/lib/python3.12/site-packages
Requires: paddlex, PyYAML, requests, typing-extensions
Required-by: 
GPU available: True


In [8]:
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I0416 05:20:16.847527  3602 pir_interpreter.cc:1524] New Executor is Running ...
W0416 05:20:16.849215  3602 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.9
I0416 05:20:16.850010  3602 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...


PaddlePaddle works well on 1 GPU.


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_cusolver_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/cusolver/lib', default_value='')
FLAGS(name='FLAGS_nccl_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/nccl/lib', default_value='')
FLAGS(name='FL

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


detection_model='/kaggle/working/models/detection'
recognition_model='/kaggle/working/models/recognition'

In [9]:
!python -m pip install -U scikit-image albumentations transformers accelerate

  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.3.0.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.7.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.7.1.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.5.4.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_

In [23]:
!python PaddleOCR/tools/infer/predict_det.py \
    --det_algorithm="DB"\
    --det_model_dir="/kaggle/input/models/det" \
    --image_dir="/kaggle/input/testdata/test.jpg" \
    --use_gpu=False\
    --enable_mkldnn=False\
    --use_angle_cls=True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/04/16 05:26:37] ppocr INFO: test.jpg	[[[502.0, 1672.0], [568.0, 1672.0], [568.0, 1696.0], [502.0, 1696.0]], [[441.0, 1672.0], [498.0, 1672.0], [498.0, 1696.0], [441.0, 1696.0]], [[370.0, 1669.0], [442.0, 1674.0], [439.0, 1701.0], [368.0, 1695.0]], [[857.0, 1669.0], [927.0, 1669.0], [927.0, 1694.0], [857.0, 1694.0]], [[757.0, 1669.0], [848.0, 1669.0], [848.0, 1694.0], [757.0, 1694.0]], [[668.0, 1669.0], [750.0, 1669.0], [750.0, 1694.0], [668.0, 1694.0]], [[620.0, 1669.0], [659.0, 1669.0], [659.0, 1694.0], [620.0, 1694.0]], [[570.0, 1669.0], [623.0, 1669.0], [623.0, 1694.0], [570.0, 1694.0]], [[314.0, 1669.0], [364.0, 1669.0], [364.0, 1696.0], [314.0, 1696.0]], [[4

In [24]:
def center_img(result_test):
    centers_dict = {}
    for item in result_test:
        points = item['points']
        text = item['transcription']
        x_center = (points[0][0] + points[2][0]) / 2
        y_center = (points[0][1] + points[2][1]) / 2

        # Thêm vào dictionary với key là points và value là [x_center, y_center, text]
        centers_dict[tuple(map(tuple, points))] = [x_center, y_center, text]

    return centers_dict


In [25]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from matplotlib.patches import Polygon

def expand_bboxes(bb_centers, eps):
    # Chuyển đổi dictionary bb_centers thành array để sử dụng cho DBSCAN
    bb = np.array(list(bb_centers.keys()))       # Tất cả bounding boxes
    centers = np.array([value[:2] for value in bb_centers.values()])  # Tọa độ trung tâm
    texts = [value[2] for value in bb_centers.values()]  # Văn bản

    # Áp dụng DBSCAN để phân nhóm các bounding boxes dựa trên tọa độ trung tâm
    db = DBSCAN(eps=eps, min_samples=1).fit(centers)
    labels = db.labels_

    # Tạo dictionary để lưu các nhóm bounding boxes và văn bản tương ứng
    clusters = {}
    for i, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = {'bboxes': [], 'texts': []}
        clusters[label]['bboxes'].append(bb[i])
        clusters[label]['texts'].append(texts[i])

    # Kết hợp bounding boxes và text cho mỗi nhóm
    combined_results = []
    for label in clusters:
        bboxes = clusters[label]['bboxes']
        texts = clusters[label]['texts']

        # Kết hợp văn bản và bounding boxes
        text_str = ' '.join(texts)  # Nối tất cả văn bản lại với nhau
        combined_results.append([text_str, bboxes])

    return combined_results

In [14]:
import json
def run(img_path, img_name):
    run_predict(img_path)
    with open('ocr_results.json', 'r') as f:
        result_test = json.load(f)
    output_list = []
    if result_test == []: 
        with open(f"{img_name}.json", 'w', encoding='utf-8') as json_file:
             json.dump(output_list, json_file, ensure_ascii=False, indent=4)
    else:           
        label_dict = expand_bboxes(center_img(result_test), eps=100)
        for item in label_dict:
            output_list.append(item[0])
        with open(f"{img_name}.json", 'w', encoding='utf-8') as json_file:
            json.dump(output_list, json_file, ensure_ascii=False, indent=4)

In [26]:
import os
import re
from tqdm.notebook import tqdm  # Use this in Jupyter environments



def natural_sort_key(s):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', s)]

input_path = "/kaggle/input/testdata"
output_path = "/keyframe_ocr"

if not os.path.exists(output_path):
    os.makedirs(output_path)

for root, dirs, files in os.walk(input_path):
    dirs.sort(key=natural_sort_key)
    for vxx_dir in tqdm(dirs, desc="Processing directories", leave=False):
        #if vxx_dir != "V001": break
        vxx_path = os.path.join(root, vxx_dir)
        vxx_output_path = os.path.join(output_path, os.path.relpath(root, input_path), vxx_dir)

        if not os.path.exists(vxx_output_path):
            os.makedirs(vxx_output_path)

        files = sorted(os.listdir(vxx_path))
        for file in tqdm(files, desc=f"Processing files in {vxx_dir}", leave=False):
            #if file != "000000.jpg": continue
            if file.lower().endswith('.jpg'):
                file_path = os.path.join(vxx_path, file)
                output_file = os.path.join(vxx_output_path, file[:-4])
                run(file_path, output_file)

print("Processing complete.")


Processing directories: 0it [00:00, ?it/s]

Processing complete.


In [27]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np

def draw_boxes(image_path, ocr_results, output_path, flag):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("/content/1942.ttf", 40)
        except IOError:
            font = ImageFont.load_default()

        if flag == 1:
            for item in ocr_results:
                flat_bbox = [coord for point in item['points'] for coord in point]
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)
        else:
            for item in ocr_results:
                all_points = np.vstack(item[1])
                text = item[0]
                print(text)

                # Calculate the bounding box
                x_min, y_min = np.min(all_points, axis=0)
                x_max, y_max = np.max(all_points, axis=0)

                expanded_bbox = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])

                # Flatten the bounding box for drawing
                flat_bbox = expanded_bbox.flatten().tolist()
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)

                text_position = (x_min, max(0, y_min - 20))
                draw.text(text_position, text, font=font, fill=(0, 255, 0))  # Green text

        img.save(output_path)

        img.show()  # This will open the image with the drawn text and boxes

In [28]:
def run_predict(image_dir):
    #os.chdir('/kaggle/working/ppocr-vietnamese')
    command = f"""
    python PaddleOCR/tools/infer/predict_system.py \\
        --image_dir="{image_dir}" \\
        --det_model_dir="inference/det" \\
        --det_algorithm="DB" \\
        --rec_image_shape="3, 48, 320" \\
        --rec_model_dir="inference/rec" \\
        --rec_char_dict_path="vn_dictionary.txt" \\
        --vis_font_path="vietnam-light.ttf" \\
        --drop_score 0.6 \
        --use_gpu USE_GPU
    """
    os.system(command)

In [18]:
!rm -r inference_results

In [19]:
# First, uncomment and use the draw_boxes function
from PIL import Image, ImageDraw, ImageFont
import numpy as np

def draw_boxes(image_path, ocr_results, output_path, flag):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("vietnam-light.ttf", 40)  # Use available font
        except IOError:
            font = ImageFont.load_default()

        if flag == 1:
            for item in ocr_results:
                flat_bbox = [coord for point in item['points'] for coord in point]
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)
        else:
            for item in ocr_results:
                all_points = np.vstack(item[1])
                text = item[0]
                print(text)

                # Calculate the bounding box
                x_min, y_min = np.min(all_points, axis=0)
                x_max, y_max = np.max(all_points, axis=0)

                expanded_bbox = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])

                # Flatten the bounding box for drawing
                flat_bbox = expanded_bbox.flatten().tolist()
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)

                text_position = (x_min, max(0, y_min - 20))
                draw.text(text_position, text, font=font, fill=(0, 255, 0))

        img.save(output_path)
        img.show()

In [20]:
!rm ocr_result.json

rm: cannot remove 'ocr_result.json': No such file or directory


In [4]:
import json
import subprocess
import sys
import re

def run_ocr(image_path):
    """Run OCR and return results as JSON with labels and bounding boxes."""
    
    command = [
        sys.executable,  # Use current Python interpreter
        "/kaggle/working/PaddleOCR/tools/infer/predict_system.py",
        "--det_algorithm=DB",
        "--det_model_dir=/kaggle/input/models/det",
        "--rec_model_dir=/kaggle/input/recognize/rec",
        f"--image_dir={image_path}",
        "--use_gpu=True",
        "--rec_char_dict_path=/kaggle/input/utils-model/vn_dictionary.txt" 
        # "--enable_mkldnn=False",
        # "--show_log=False",  # Disable verbose logging for speed
        # "--warmup=False",     # Skip warmup for faster execution
        # "--rec_batch_num=32",
        # "--cpu_threads=16",
        
    ]
    
    # Run with subprocess for better control and error handling
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            # timeout=30  # 30 second timeout
        )
        
        if result.returncode != 0:
            print(f"OCR Error: {result.stderr}")
            return [], {}
        
        # Extract timing information from stdout
        timing_info = extract_timing_info(result.stdout)
            
    except subprocess.TimeoutExpired:
        print("OCR process timeout after 30 seconds")
        return [], {}
    except Exception as e:
        print(f"OCR failed: {e}")
        return [], {}
    
    # Load the JSON results
    try:
        with open('ocr_results.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print("OCR results file not found")
        return [], {}
    except json.JSONDecodeError:
        print("Invalid JSON in OCR results")
        return [], {}
    
    # Transform the data to add label and bounding_box fields
    results = []
    for item in data.get('root', []):
        points = item['points']
        results.append({
            'transcription': item['transcription'],
            'label': item['transcription'],
            'points': points,
            'bounding_box': {
                'top_left': points[0],
                'top_right': points[1],
                'bottom_right': points[2],
                'bottom_left': points[3]
            }
        })
    
    return results, timing_info


def extract_timing_info(stdout):
    """Extract timing information from stdout."""
    timing_info = {}
    
    # Patterns to match timing lines
    patterns = {
        'detection': r'Detection time:\s+([\d.]+) seconds',
        'classification': r'Classification time:\s+([\d.]+) seconds',
        'recognition': r'Recognition time:\s+([\d.]+) seconds',
        'total_inference': r'Total inference time:\s+([\d.]+) seconds',
        'overall': r'Overall time:\s+([\d.]+) seconds'
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, stdout)
        if match:
            timing_info[key] = float(match.group(1))
    
    return timing_info


def print_timing_info(timing_info):
    """Print timing information in a formatted way."""
    if not timing_info:
        return
    
    print("\n" + "="*50)
    print("TIMING STATISTICS")
    print("="*50)
    
    if 'detection' in timing_info:
        print(f"Detection time:       {timing_info['detection']:.4f} seconds")
    
    if 'classification' in timing_info:
        print(f"Classification time:  {timing_info['classification']:.4f} seconds")
    
    if 'recognition' in timing_info:
        print(f"Recognition time:     {timing_info['recognition']:.4f} seconds")
    
    if 'total_inference' in timing_info:
        print(f"Total inference time: {timing_info['total_inference']:.4f} seconds")
    
    if 'overall' in timing_info:
        print(f"Overall time:         {timing_info['overall']:.4f} seconds")
    
    print("="*50 + "\n")


# Usage:
if __name__ == "__main__":
    ocr_results, timing_info = run_ocr("/kaggle/input/testdata/11.jpeg")
    
    # Print timing information
    print_timing_info(timing_info)
    
    if ocr_results:
        print(f"✓ Found {len(ocr_results)} text regions\n")
        
        # Print results
        for i, item in enumerate(ocr_results, 1):
            print(f"{i}. Label: {item['label']}")
            print(f"   Bounding Box: {item['bounding_box']}")
            print("---")
    else:
        print("No text detected or OCR failed")


TIMING STATISTICS
Detection time:       0.5107 seconds
Recognition time:     0.9367 seconds
Total inference time: 1.4474 seconds
Overall time:         1.5323 seconds

✓ Found 194 text regions

1. Label: VIỆN
   Bounding Box: {'top_left': [457, 21], 'top_right': [530, 21], 'bottom_right': [530, 51], 'bottom_left': [457, 51]}
---
2. Label: ĐA
   Bounding Box: {'top_left': [532, 21], 'top_right': [574, 21], 'bottom_right': [574, 45], 'bottom_left': [532, 45]}
---
3. Label: KHOA
   Bounding Box: {'top_left': [576, 18], 'top_right': [667, 12], 'bottom_right': [669, 42], 'bottom_left': [577, 47]}
---
4. Label: GIA
   Bounding Box: {'top_left': [670, 13], 'top_right': [727, 13], 'bottom_right': [727, 45], 'bottom_left': [670, 45]}
---
5. Label: ĐỊNH
   Bounding Box: {'top_left': [729, 13], 'top_right': [809, 13], 'bottom_right': [809, 43], 'bottom_left': [729, 43]}
---
6. Label: BỆNH
   Bounding Box: {'top_left': [368, 26], 'top_right': [451, 21], 'bottom_right': [453, 51], 'bottom_left': [3

In [21]:
import gc
import torch

# 1. Delete references
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

# 2. Force Garbage Collection
gc.collect()

# 3. Clear CUDA cache
with torch.no_grad():
    torch.cuda.empty_cache()

print("✅ GPU Memory Cleared!")


✅ GPU Memory Cleared!


In [11]:
!pip install -q -U bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01


In [22]:
import json
import torch
import time
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Dict, Optional

# ==================== 1. INITIALIZE LOCAL LLM ====================
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Check if model is already loaded to avoid OOM errors when re-running cells
if "model" not in globals():
    print(f"Loading {MODEL_ID} into Kaggle GPU...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("✓ Local model loaded successfully!")
else:
    print("⚡ Model already in memory, skipping reload.")

# ==================== 2. PROCESSING FUNCTION ====================

def ocr_to_structured_json(ocr_results: List[Dict]) -> Optional[Dict]:
    if not ocr_results:
        print("Error: No OCR results to process")
        return None
    
    ocr_text = "\n".join([f"- {item.get('label', item.get('transcription', ''))}" for item in ocr_results])
    
    prompt = f"""You are an expert at extracting structured data from Vietnamese medicine invoices.
Given the OCR text, extract and return a JSON object. Correct Vietnamese spelling errors found in OCR.

Structure:
{{
  "patient_name": "string or null",
  "doctor_name": "string or null",
  "clinic_hospital": "string or null",
  "date": "DD/MM/YYYY or null",
  "diagnosis": "string or null",
  "medicines": [
    {{ "name": "name", "dosage": "amount", "quantity": "qty", "frequency": "freq", "duration": "days", "instructions": "notes" }}
  ],
  "notes": "string or null"
}}

OCR Text:
{ocr_text}

IMPORTANT: Return ONLY valid JSON."""

    messages = [
        {"role": "system", "content": "You are a helpful data extraction assistant. You only output valid JSON."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    print("Generating structured data locally...")
    start_time = time.time() # Start timing
    
    try:
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=500,
            temperature=0.1,
            use_cache=True,
            do_sample=False,
        )
        
        # Calculate time taken
        end_time = time.time()
        duration = end_time - start_time
        
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        llm_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        # Robust JSON cleaning
        llm_response = llm_response.replace("```json", "").replace("```", "").strip()
            
        structured_data = json.loads(llm_response)
        
        print(f"✓ Successfully converted in {duration:.2f} seconds") # Print timing here
        return structured_data
        
    except Exception as e:
        print(f"Error: {e}")
        return None

# ==================== TEST BLOCK ====================

if __name__ == "__main__":
    # Pointing to the generated OCR results file
    # (Checking for both ocr_results.json and orc_result.json just in case)
    json_path = "./ocr_results.json" if os.path.exists("./ocr_results.json") else "./ocr_result.json"
    
    if os.path.exists(json_path):
        print(f"Loading data from {json_path}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        # Convert PaddleOCR format (root list) to the format expected by LLM
        # Handle both list and nested {root: []} formats
        if isinstance(raw_data, dict) and 'root' in raw_data:
            ocr_results = [{"label": item['transcription']} for item in raw_data['root']]
        elif isinstance(raw_data, list):
            ocr_results = [{"label": item.get('transcription', item.get('label', ''))} for item in raw_data]
        else:
            ocr_results = []
            
        print(f"Extracted {len(ocr_results)} text lines.")
        
        # Run conversion
        structured_invoice = ocr_to_structured_json(ocr_results)
        
        if structured_invoice:
            print("\n" + "="*50)
            print("STRUCTURED MEDICINE INVOICE")
            print("="*50)
            print(json.dumps(structured_invoice, indent=2, ensure_ascii=False))
    else:
        print(f"❌ File not found at {json_path}. Please run the OCR cell first.")


Loading Qwen/Qwen2.5-3B-Instruct into Kaggle GPU...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✓ Local model loaded successfully!
Loading data from ./ocr_results.json...
Extracted 194 text lines.
Generating structured data locally...
✓ Successfully converted in 17.99 seconds

STRUCTURED MEDICINE INVOICE
{
  "patient_name": "Anh Đào",
  "doctor_name": "Khám bệnh",
  "clinic_hospital": "Bệnh viện đa khoa Gia Định",
  "date": "31/07/2025",
  "diagnosis": "Bệnh tăng huyết áp (nguyên phát)",
  "medicines": [
    {
      "name": "acid clavulanic 125mg",
      "dosage": "125mg",
      "quantity": "10 viên",
      "frequency": "Sáng",
      "duration": "20 ngày",
      "instructions": "(Sau khi ăn no)"
    },
    {
      "name": "Amoxicillin 875mg",
      "dosage": "875mg",
      "quantity": "2 viên",
      "frequency": "Sáng, Chiều, Tối",
      "duration": "20 ngày",
      "instructions": "(Sau khi ăn no)"
    },
    {
      "name": "Paracetamol 650mg",
      "dosage": "650mg",
      "quantity": "2 viên",
      "frequency": "Sáng, Chiều, Tối",
      "duration": "20 ngày",
      "instru

In [ ]:
# ==============================================================================
# OPTIMIZED MASTER BRIDGE (WARM START OCR + WARM START LLM)
# ==============================================================================
import os
import sys
import json
import torch
import time
import re
import subprocess
import threading
import cv2
import numpy as np
from typing import List, Dict, Optional
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. SETUP ENVIRONMENT & DEPENDENCIES
if not os.path.exists("cloudflared"):
    print("Downloading cloudflared...")
    os.system("curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared")
    os.system("chmod +x cloudflared")

# Path setup for PaddleOCR
PADDLE_PATH = "/kaggle/working/PaddleOCR"
if PADDLE_PATH not in sys.path:
    sys.path.append(PADDLE_PATH)
    sys.path.append(os.path.join(PADDLE_PATH, "tools/infer"))

# 2. GLOBAL INITIALIZATION (WARM START)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# --- Initialize LLM (Smart Skip) ---
if "model" not in globals():
    print("Initializing LLM...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # Using float16 for best speed on T4
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    print("✓ LLM Ready")
else:
    print("⚡ LLM already in memory")

# --- Initialize OCR (Speed Fix: Persistent Engine) ---
# --- Initialize OCR (Speed Fix: Persistent Engine) ---
# --- Initialize OCR (Speed Fix: Persistent Engine) ---
if "text_sys" not in globals():
    print("Initializing PaddleOCR Engine...")
    import predict_system
    import utility
    import sys
    
    # Temporarily hide Jupyter arguments to avoid the crash
    old_argv = sys.argv
    sys.argv = [old_argv[0]] 
    try:
        args = utility.parse_args()
    finally:
        sys.argv = old_argv # Restore arguments for the rest of the notebook
    
    # Configure required paths
    args.det_model_dir = "/kaggle/input/models/det"
    args.rec_model_dir = "/kaggle/input/recognize/rec"
    args.rec_char_dict_path = "/kaggle/input/utils-model/vn_dictionary.txt"
    args.use_gpu = True
    
    text_sys = predict_system.TextSystem(args)
    print("✓ OCR Engine Ready")
else:
    print("⚡ OCR Engine already in memory")

# 3. CORE PROCESSING LOGIC
def run_ocr_fast(image_path):
    img = cv2.imread(image_path)
    if img is None: return []
    
    start_time = time.time()
    # Library call (Zero cold-start delay)
    ocr_output = text_sys(img)
    dt_boxes = ocr_output[0]
    rec_res = ocr_output[1] 
    duration = time.time() - start_time
    
    results = []
    for i in range(len(dt_boxes)):
        results.append({
            'label': rec_res[i][0],
            'confidence': float(rec_res[i][1]),
            'points': dt_boxes[i].tolist()
        })
    print(f"⚡ Raw OCR Inference: {duration:.2f}s")
    return results

def ocr_to_structured_json(ocr_results: List[Dict]) -> Optional[Dict]:
    if not ocr_results: return None
    
    # 1. Prepare OCR text
    ocr_text = "\n".join([f"- {item.get('label', '')}" for item in ocr_results])
    
    # 2. Strict Schema Prompt
    prompt = f"""You are a Vietnamese medical data extractor. 
Convert the OCR text below into a JSON object with this EXACT structure:
{{
  "patient_name": "fullName or null",
  "doctor_name": "doctorName or null",
  "clinic_hospital": "clinic/hospital name or null",
  "date": "DD/MM/YYYY or null",
  "diagnosis": "diagnosis or null",
  "medicines": [
    {{
      "name": "medicine name",
      "dosage": "e.g. 500mg",
      "quantity": "count",
      "frequency": "how often",
      "duration": "days",
      "instructions": "how to take"
    }}
  ],
  "notes": "additional info or null"
}}
OCR Text:
{ocr_text}
IMPORTANT: 
- Return ONLY valid JSON.
- Fix Vietnamese spelling errors.
- Ensure all medicines are extracted correctly."""
    messages = [
        {"role": "system", "content": "You are a helpful medical assistant. You only output valid JSON with specific keys."},
        {"role": "user", "content": prompt}
    ]
    
    # Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=1000, # Increased for complex prescriptions
            do_sample=False, 
            use_cache=True
        )
    
    input_len = model_inputs.input_ids.shape[1]
    response = tokenizer.decode(generated_ids[0][input_len:], skip_special_tokens=True).strip()
    
    print(f"⏱️ LLM Extraction: {time.time() - start_time:.2f}s")
    
    try:
        # Clean JSON markdown blocks
        json_str = re.sub(r'^```json\s*|\s*```$', '', response, flags=re.MULTILINE)
        return json.loads(json_str)
    except Exception as e:
        print(f"JSON Parse Error: {e}")
        print(f"Raw response: {response}")
        return None
        
# 4. API & TUNNEL GATEWAY
from fastapi import FastAPI, File, UploadFile
app = FastAPI()

@app.post("/ocr")
async def process_ocr(file: UploadFile = File(...)):
    tmp_path = "/kaggle/working/temp_scan.jpg"
    with open(tmp_path, "wb") as b: b.write(await file.read())
    try:
        results = run_ocr_fast(tmp_path)
        if not results: return JSONResponse(status_code=500, content={"error": "OCR failed"})
        return ocr_to_structured_json(results)
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/health")
def health(): return {"status": "ready"}

# Kill old servers/tunnels
!fuser -k 8000/tcp || true
!pkill cloudflared || true

# Start API in background
threading.Thread(target=lambda: __import__("uvicorn").run(app, host="0.0.0.0", port=8000, log_level="error"), daemon=True).start()
time.sleep(3)

# Start Tunnel
print("\n" + "="*60 + "\n  ESTABLISHING PUBLIC GATEWAY...\n" + "="*60)
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

found_url = False
try:
    for line in tunnel_proc.stdout:
        clean_line = re.sub(r'\x1b\[[0-9;]*m', '', line)
        if "trycloudflare.com" in clean_line and "https://" in clean_line:
            match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', clean_line)
            if match:
                url = match.group(1)
                print(f"\n🚀 OCR GATEWAY IS LIVE!")
                print(f"👉 COPY THIS TO YOUR .env: {url}\n")
                found_url = True
        if found_url and "GET /health" not in clean_line:
            print(clean_line.strip())
except KeyboardInterrupt:
    print("\nStopping tunnel...")
    tunnel_proc.terminate()


Initializing LLM...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✓ LLM Ready
Initializing PaddleOCR Engine...


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


[2026/04/16 06:38:02] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/04/16 06:38:04] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
✓ OCR Engine Ready

  ESTABLISHING PUBLIC GATEWAY...

🚀 OCR GATEWAY IS LIVE!
👉 COPY THIS TO YOUR .env: https://signature-talking-sodium-obj.trycloudflare.com

2026-04-16T06:38:12Z INF |  https://signature-talking-sodium-obj.trycloudflare.com                                    |
2026-04-16T06:38:12Z INF +--------------------------------------------------------------------------------------------+
2026-04-16T06:38:12Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-04-16T06:38:12Z INF Version 2026.3.0 (Checksum 4a9e50e6d6d798e90fcd01933151a90bf7edd99a0a55c28ad18f2e16263a5c30)
2026-04-16T06:38:12Z INF GOOS: linux, GOVersion: go1.24.13, GoArch: amd64
2026-04-16T06:3

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚡ Raw OCR Inference: 1.59s
⏱️ LLM Extraction: 26.91s
⚡ Raw OCR Inference: 0.89s
⏱️ LLM Extraction: 27.98s
⚡ Raw OCR Inference: 1.10s
⏱️ LLM Extraction: 38.66s
⚡ Raw OCR Inference: 1.50s
⏱️ LLM Extraction: 25.85s
⚡ Raw OCR Inference: 0.86s
⏱️ LLM Extraction: 27.94s
⚡ Raw OCR Inference: 0.86s
⏱️ LLM Extraction: 27.85s
